目标：请实现 rotate_image 函数：

#### 参数
image (numpy 数组): 输入的灰度图像，包含原始像素值。
angle (浮点数): 旋转角度，以度为单位。正值表示逆时针旋转，负值表示顺时针旋转。

#### 功能
将输入的灰度图像按指定角度旋转。
使用最近邻插值来确定旋转后图像中每个像素的值。
旋转后的图像应该包含原图像的所有部分，因此可能需要调整输出图像的大小。

#### 返回值

rotated_image (numpy 数组): 旋转后的图像像素值。

In [4]:
import numpy as np

In [5]:
def rotate_image(image, angle):
# 1. 角度标准化
    angle = angle % 360
    theta = np.radians(angle)
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    # 获取原图尺寸
    h, w = image.shape
    # 几何中心（从 0 开始的索引中点）
    cx, cy = (w - 1) / 2.0, (h - 1) / 2.0

    # 2. 计算新画布尺寸
    # 为了避免 90/180/270 度时的浮点误差导致多出 1 像素，我们使用 round 处理
    new_w = int(round(abs(w * cos_t) + abs(h * sin_t)))
    new_h = int(round(abs(w * sin_t) + abs(h * cos_t)))
    
    # 特殊情况处理：如果是 360 或 0 度，直接返回原图
    if angle % 360 == 0:
        return image.copy()

    # 新图像的几何中心
    new_cx, new_cy = (new_w - 1) / 2.0, (new_h - 1) / 2.0

    # 初始化输出图像
    rotated_image = np.zeros((new_h, new_w), dtype=np.uint8)

    # 3. 逆向映射
    # 生成新图像的所有坐标索引
    y_target, x_target = np.indices((new_h, new_w))

    # 将坐标平移到以新图中心为原点
    tx = x_target - new_cx
    ty = y_target - new_cy

    # 逆向旋转变换：目标坐标 -> 原始坐标
    # 逆旋转矩阵是顺时针旋转 angle (即 cos(t), sin(t); -sin(t), cos(t))
    source_x = tx * cos_t + ty * sin_t + cx
    source_y = -tx * sin_t + ty * cos_t + cy

    # 4. 最近邻插值 (Nearest Neighbor)
    # 使用 np.round 找到最接近的像素点坐标
    source_x_int = np.round(source_x).astype(int)
    source_y_int = np.round(source_y).astype(int)

    # 5. 边界检查
    # 只有落在原图坐标范围 [0, w-1] 和 [0, h-1] 内的点才赋值
    mask = (source_x_int >= 0) & (source_x_int < w) & \
           (source_y_int >= 0) & (source_y_int < h)

    # 填充像素
    rotated_image[y_target[mask], x_target[mask]] = image[source_y_int[mask], source_x_int[mask]]

    return rotated_image
    # TODO
    



In [6]:
def main():
    # 5x5 的灰度图像
    image = np.array([[100, 150, 200, 50, 75],
                      [80, 120, 160, 40, 60],
                      [180, 210, 240, 90, 110],
                      [20, 30, 40, 10, 15],
                      [130, 170, 220, 70, 95]], dtype=np.uint8)
    # 旋转 90 度
    rotated_image = rotate_image(image, 90)
    print(rotated_image)

if __name__ == '__main__':
    main()

[[130  20 180  80 100]
 [170  30 210 120 150]
 [220  40 240 160 200]
 [ 70  10  90  40  50]
 [ 95  15 110  60  75]]
